# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The paper reports that the proposed model improves prediction performance compared with the baseline.

**My methodology question:**

Where do the labels come from? Were they created manually, automatically, or using heuristic rules? Knowing this would help understand how reliable the evaluation is.

---

## Finding 2

The paper evaluates the model using a validation split.

**My methodology question:**

Does the validation design prevent data leakage? For example, are pages from the same client kept together, or could similar examples appear in both the training and test sets?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
# ==========================================
# Week 6 - Honest Split (Before / After)
# ==========================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------
# Load Dataset
# ------------------------------------------

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# ------------------------------------------
# Recreate Week 4 Baseline Score
# ------------------------------------------

stale = df["days_since_last_update"] >= 180

visible = df["impressions_90d"] >= 1000

ctr_gap = (
    (df["avg_position"] > 0)
    &
    (df["avg_position"] <= 10)
    &
    (df["ctr"] < 2)
)

df["baseline_score"] = (
      2 * stale.astype(int)
    + 2 * visible.astype(int)
    + 3 * ctr_gap.astype(int)
)

# ------------------------------------------
# Target
# ------------------------------------------

df["target"] = (df["baseline_score"] >= 5).astype(int)

# ------------------------------------------
# Features
# ------------------------------------------

X = df[
    [
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
]

y = df["target"]

# ------------------------------------------
# Split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ------------------------------------------
# Week 4 Rule Baseline
# ------------------------------------------

baseline_pred = (
    (
        2 * (X_test["days_since_last_update"] >= 180).astype(int)
        +
        2 * (X_test["impressions_90d"] >= 1000).astype(int)
        +
        3 * (
            (
                (X_test["avg_position"] > 0)
                &
                (X_test["avg_position"] <= 10)
                &
                (X_test["ctr"] < 2)
            ).astype(int)
        )
    ) >= 5
).astype(int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

# ------------------------------------------
# Random Forest
# ------------------------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

rf_pred = model.predict(X_test)

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

# ------------------------------------------
# Comparison Table
# ------------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "Week 4 Rule Baseline",
        "Week 5 Random Forest"
    ],
    "Accuracy": [
        baseline_accuracy,
        rf_accuracy
    ]
})

print(comparison)
comparison

                  Model  Accuracy
0  Week 4 Rule Baseline       1.0
1  Week 5 Random Forest       1.0


,Model,Accuracy
0,Week 4 Rule Baseline,1.0
1,Week 5 Random Forest,1.0


### Before

The Week 5 model used a stratified random train/test split.

### Honest split audit

The available modeling dataset contains only the selected numerical features and does not include a client identifier or timestamp.

Because of this limitation, I could not perform grouped-by-client or time-aware validation.

If those fields were available, I would re-run the model using grouped or time-aware validation to obtain a more realistic estimate of generalization.

Therefore, the Week 5 accuracy should be interpreted as measured under a stratified random split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

I checked my final feature set for possible data leakage.

- The target column was not used as a feature.
- The model only uses four numerical features:
  - days_since_last_update
  - impressions_90d
  - avg_position
  - ctr
- No future information was included in the features.
- The dataset does not contain client or timestamp fields, so grouped or time-aware validation could not be performed.

Based on this review, I did not find obvious feature leakage in my final model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Original claim

The Random Forest model accurately predicts content refresh opportunities.

## Revised public-safe claim

In this notebook, the Random Forest model showed strong predictive performance on the available evaluation dataset.

The reported accuracy was measured using a stratified train/test split. Because the dataset does not include client identifiers or timestamps, grouped or time-aware validation could not be performed.

These findings should be interpreted as decision-support evidence rather than proof that the model will generalize to every future dataset.

## 5. Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.